<a href="https://colab.research.google.com/github/AkankshaB123/Optimisation/blob/main/bayesian_optimization_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install optuna

In [14]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Load the Pima Indian Diabetes dataset from a reliable mirror
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [15]:
import numpy as np

#Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose','BloodPressure', 'SkinThickness','Insulin','BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

#Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

#Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [16]:
#Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

#Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Check the shape of the data
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (614, 8)
y_train shape: (614,)
X_test shape: (154, 8)
y_test shape: (154,)


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [24]:
def objective(trial):
    # Define the hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    # Initialize the model with suggested hyperparameters
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    # Evaluate the model using 5-fold cross-validation
    score = cross_val_score(clf, X_train, y_train, n_jobs=-1, cv=5).mean()
    return score

# Create and run the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best Trial:")
print("  Value (CV Accuracy): ", study.best_trial.value)
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-10-04 17:20:05,469] A new study created in memory with name: no-name-56b2d877-ced2-4b44-8e12-585340383d5d
[I 2026-10-04 17:20:07,228] Trial 0 finished with value: 0.7801679328268692 and parameters: {'n_estimators': 16, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.7801679328268692.
[I 2026-10-04 17:20:07,382] Trial 1 finished with value: 0.7785685725709716 and parameters: {'n_estimators': 19, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.7801679328268692.
[I 2026-10-04 17:20:07,869] Trial 2 finished with value: 0.7687591630014661 and parameters: {'n_estimators': 80, 'max_depth': 5, 'min_samples_split': 3, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.7801679328268692.
[I 2026-10-04 17:20:08,571] Trial 3 finished with value: 0.7540850326536052 and parameters: {'n_estimators': 119, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.78

Best Trial:
  Value (CV Accuracy):  0.7899373583899774
  Params: 
    n_estimators: 21
    max_depth: 24
    min_samples_split: 8
    min_samples_leaf: 4


In [19]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize') # We aim to maximize accuracy
study.optimize(objective, n_trials=50) #Run 50 trials to find the best hyperparameters

[I 2026-10-04 17:10:42,776] A new study created in memory with name: no-name-c748a3b9-ffef-4d88-ba60-275569dfcfe8
[I 2026-10-04 17:10:44,109] Trial 0 finished with value: 0.7655204584832733 and parameters: {'n_estimators': 140, 'max_depth': 14, 'min_samples_split': 4, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.7655204584832733.
[I 2026-10-04 17:10:44,716] Trial 1 finished with value: 0.7247634279621484 and parameters: {'n_estimators': 53, 'max_depth': 2, 'min_samples_split': 7, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.7655204584832733.
[I 2026-10-04 17:10:46,934] Trial 2 finished with value: 0.7671464747434359 and parameters: {'n_estimators': 180, 'max_depth': 18, 'min_samples_split': 3, 'min_samples_leaf': 10}. Best is trial 2 with value: 0.7671464747434359.
[I 2026-10-04 17:10:47,593] Trial 3 finished with value: 0.7655071304811409 and parameters: {'n_estimators': 50, 'max_depth': 23, 'min_samples_split': 3, 'min_samples_leaf': 7}. Best is trial 2 with value: 

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def Objective(trial):
  # Suggest values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )

  #Perform 3-fold cross-validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  return score #Return the accuracy score for Optuna to maximize

In [21]:
#Print the test result
print(f'Best trial accuracy:{study.best_trial.value}')
print(f'Best trial parameters:{study.best_trial.params}')

Best trial accuracy:0.7915900306544048
Best trial parameters:{'n_estimators': 20, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 4}


In [22]:
from sklearn.metrics import accuracy_score

#Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

#Fit the model to the training data
best_model.fit(X_train, y_train)

#Make predictions on the test set
y_pred = best_model.predict(X_test)

#Calculate and print the accuracy of the model on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy}')

Test Accuracy: 0.7727272727272727


In [25]:
#Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(Objective, n_trials=50)

[I 2026-10-04 17:20:25,651] A new study created in memory with name: no-name-dfb75b21-bac4-42ea-87ad-9dc4fab19405
[I 2026-10-04 17:20:25,968] Trial 0 finished with value: 0.7507970667941973 and parameters: {'n_estimators': 85, 'max_depth': 3}. Best is trial 0 with value: 0.7507970667941973.
[I 2026-10-04 17:20:26,449] Trial 1 finished with value: 0.7768770923003347 and parameters: {'n_estimators': 107, 'max_depth': 18}. Best is trial 1 with value: 0.7768770923003347.
[I 2026-10-04 17:20:27,010] Trial 2 finished with value: 0.7573250438386737 and parameters: {'n_estimators': 142, 'max_depth': 4}. Best is trial 1 with value: 0.7768770923003347.
[I 2026-10-04 17:20:27,511] Trial 3 finished with value: 0.771975131516021 and parameters: {'n_estimators': 118, 'max_depth': 6}. Best is trial 1 with value: 0.7768770923003347.
[I 2026-10-04 17:20:28,018] Trial 4 finished with value: 0.780121154152718 and parameters: {'n_estimators': 118, 'max_depth': 16}. Best is trial 4 with value: 0.7801211541

In [26]:
# Print the best result
print(f'Best trial accuracy:{study.best_trial.value}')
print(f'Best trial parameters:{study.best_trial.params}')

Best trial accuracy:0.7866331898613104
Best trial parameters:{'n_estimators': 160, 'max_depth': 13}


In [28]:
from sklearn.metrics import accuracy_score

#Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

#Fit the model to the training data
best_model.fit(X_train, y_train)

#Make predictions on the test set
y_pred = best_model.predict(X_test)

#Calculate and print the accuracy of the model on the test set
accuracy = accuracy_score(y_test, y_pred)

#Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


In [29]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 5, 7, 10, 15, 20]
}

In [30]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(Objective)

[I 2026-10-04 17:28:36,861] A new study created in memory with name: no-name-d39692f5-898b-4384-91c6-5f5a783df829
[I 2026-10-04 17:28:38,785] Trial 0 finished with value: 0.7638291088793241 and parameters: {'n_estimators': 200, 'max_depth': 7}. Best is trial 0 with value: 0.7638291088793241.
[I 2026-10-04 17:28:39,856] Trial 1 finished with value: 0.7687151283277539 and parameters: {'n_estimators': 150, 'max_depth': 7}. Best is trial 1 with value: 0.7687151283277539.
[I 2026-10-04 17:28:40,976] Trial 2 finished with value: 0.7719591901801371 and parameters: {'n_estimators': 150, 'max_depth': 20}. Best is trial 2 with value: 0.7719591901801371.
[I 2026-10-04 17:28:42,097] Trial 3 finished with value: 0.7735772357723577 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 3 with value: 0.7735772357723577.
[I 2026-10-04 17:28:42,465] Trial 4 finished with value: 0.7752112227004623 and parameters: {'n_estimators': 50, 'max_depth': 20}. Best is trial 4 with value: 0.7752112

In [31]:
#Print the best result
print(f'Best trial accuracy:{study.best_trial.value}')
print(f'Best trial parameters:{study.best_trial.params}')

Best trial accuracy:0.7817391997449387
Best trial parameters:{'n_estimators': 50, 'max_depth': 10}


In [33]:
from sklearn.metrics import accuracy_score

#Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

#Fit the model to the training data
best_model.fit(X_train, y_train)

#Make prediction on the test set
y_pred = best_model.predict(X_test)

#Calculate and print the accuracy of the model on the test set
accuracy = accuracy_score(y_test, y_pred)

#Print the test accuracy
print(f'Test Accuracy with the best hyperparameters: {accuracy:.2f}')

Test Accuracy with the best hyperparameters: 0.75


# Optuna Visualizations

In [34]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [35]:
# 1. Optimization History
plot_optimization_history(study).show()

In [36]:
# 2. Parallel Coordinate Plot
plot_parallel_coordinate(study).show()

In [39]:
#3. Slice Plot
plot_slice(study).show()

In [40]:
# 4. Contour Plot
plot_contour(study).show()

In [38]:
#5. Hyperparameter Importance
plot_param_importances(study).show()